In [8]:
import os
print(os.getcwd())
print(os.path.exists(f'{DATA_DIR}/tmodel.pt'))

c:\Users\MSI 1\Documents\neural-lam-demo\gnn\gnn-weather-from-scratch
True


In [1]:
import torch
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import sys
import os
sys.path.append('.')

DATA_DIR = 'data/global'
VAR_NAMES = ['u10', 'v10', 'sp', 't850', 't500', 'z850', 'z500']

In [2]:
# Load coordinates and stats
lat = torch.load(f'{DATA_DIR}/lat.pt').numpy()  # (N_grid,) in degrees
lon = torch.load(f'{DATA_DIR}/lon.pt').numpy()  # (N_grid,) in degrees
mean = torch.load(f'{DATA_DIR}/mean.pt').numpy()  # (7,)
std  = torch.load(f'{DATA_DIR}/std.pt').numpy()   # (7,)

print(f'Grid nodes: {len(lat)}')
print(f'Lat range: {lat.min():.1f} to {lat.max():.1f}')
print(f'Lon range: {lon.min():.1f} to {lon.max():.1f}')

Grid nodes: 65160
Lat range: -90.0 to 90.0
Lon range: 0.0 to 359.0


In [3]:
def lat_lon_to_xyz(lat_deg, lon_deg, radius=1.0):
    """Convert lat/lon degrees to 3D Cartesian on a sphere of given radius."""
    lat_r = np.radians(lat_deg)
    lon_r = np.radians(lon_deg)
    x = radius * np.cos(lat_r) * np.cos(lon_r)
    y = radius * np.cos(lat_r) * np.sin(lon_r)
    z = radius * np.sin(lat_r)
    return x, y, z

def plot_globe_scatter(values, lat, lon, title, colorscale='RdBu_r', 
                       symmetric=True, marker_size=2):
    """Plot values as colored scatter on a unit sphere."""
    x, y, z = lat_lon_to_xyz(lat, lon)
    
    cmin = -np.abs(values).max() if symmetric else values.min()
    cmax =  np.abs(values).max() if symmetric else values.max()
    
    fig = go.Figure(data=go.Scatter3d(
        x=x, y=y, z=z,
        mode='markers',
        marker=dict(
            size=marker_size,
            color=values,
            colorscale=colorscale,
            cmin=cmin,
            cmax=cmax,
            colorbar=dict(title=title, thickness=15),
            showscale=True,
        )
    ))
    
    fig.update_layout(
        title=title,
        scene=dict(
            xaxis=dict(showticklabels=False, showgrid=False, zeroline=False, title=''),
            yaxis=dict(showticklabels=False, showgrid=False, zeroline=False, title=''),
            zaxis=dict(showticklabels=False, showgrid=False, zeroline=False, title=''),
            bgcolor='black',
            aspectmode='cube',
        ),
        paper_bgcolor='black',
        font_color='white',
        width=800, height=800,
    )
    return fig

def plot_globe_elevated(error, lat, lon, title, 
                         elevation_scale=0.3, colorscale='Reds'):
    """
    Plot error as elevation on a globe — nodes displaced outward
    proportional to error magnitude. Taller = larger error.
    """
    # Normalize error to [0, 1] for elevation
    err_norm = (error - error.min()) / (error.max() - error.min() + 1e-8)
    radius = 1.0 + elevation_scale * err_norm
    
    x, y, z = lat_lon_to_xyz(lat, lon, radius=radius)
    
    fig = go.Figure(data=go.Scatter3d(
        x=x, y=y, z=z,
        mode='markers',
        marker=dict(
            size=2,
            color=error,
            colorscale=colorscale,
            colorbar=dict(title='Error', thickness=15),
            showscale=True,
        )
    ))
    
    fig.update_layout(
        title=title,
        scene=dict(
            xaxis=dict(showticklabels=False, showgrid=False, zeroline=False, title=''),
            yaxis=dict(showticklabels=False, showgrid=False, zeroline=False, title=''),
            zaxis=dict(showticklabels=False, showgrid=False, zeroline=False, title=''),
            bgcolor='#0a0a1a',
            aspectmode='cube',
        ),
        paper_bgcolor='#0a0a1a',
        font_color='white',
        width=900, height=900,
    )
    return fig

In [4]:
# Load mesh node coordinates from mesh_features.pt
# mesh_features is a list of tensors, one per level: each (N_mesh[l], 2) with [lat, lon]
mesh_features = torch.load(f'{DATA_DIR}/mesh_features.pt')

print(f'Number of mesh levels: {len(mesh_features)}')
for i, mf in enumerate(mesh_features):
    print(f'  Level {i}: {mf.shape[0]} nodes')

Number of mesh levels: 3
  Level 0: 12 nodes
  Level 1: 42 nodes
  Level 2: 162 nodes


In [5]:
# Plot grid nodes + finest mesh level on globe
mesh_lat = mesh_features[-1][:, 0].numpy()
mesh_lon = mesh_features[-1][:, 1].numpy()

gx, gy, gz = lat_lon_to_xyz(lat, lon)
mx, my, mz = lat_lon_to_xyz(mesh_lat, mesh_lon, radius=1.02)  # slightly above

fig = go.Figure()

# Grid nodes
fig.add_trace(go.Scatter3d(
    x=gx, y=gy, z=gz,
    mode='markers',
    marker=dict(size=1, color='#00aaff', opacity=0.3),
    name='ERA5 grid nodes'
))

# Mesh nodes
fig.add_trace(go.Scatter3d(
    x=mx, y=my, z=mz,
    mode='markers',
    marker=dict(size=5, color='#ff4444', symbol='circle'),
    name=f'Mesh nodes (level {len(mesh_features)-1})'
))

fig.update_layout(
    title='Global Icosahedral Mesh — Grid + Mesh Nodes',
    scene=dict(
        xaxis=dict(showticklabels=False, showgrid=False, zeroline=False, title=''),
        yaxis=dict(showticklabels=False, showgrid=False, zeroline=False, title=''),
        zaxis=dict(showticklabels=False, showgrid=False, zeroline=False, title=''),
        bgcolor='#0a0a1a',
    ),
    paper_bgcolor='#0a0a1a',
    font_color='white',
    width=900, height=900,
    legend=dict(x=0.02, y=0.98)
)

fig.show()

In [6]:
# Load a few timesteps of ERA5 data
node_features = torch.load(f'{DATA_DIR}/node_features.pt')  # (T, N, 7)
print(f'Shape: {node_features.shape}')

# Pick a timestep and variable
t = 0
var_idx = 3  # t850
var_name = VAR_NAMES[var_idx]

# Denormalize
values_norm = node_features[t, :, var_idx].numpy()
values = values_norm * std[var_idx] + mean[var_idx]

fig = plot_globe_scatter(
    values=values,
    lat=lat, lon=lon,
    title=f'{var_name} at T=0 (K)',
    colorscale='RdBu_r',
    symmetric=False,
    marker_size=2
)
fig.show()

Shape: torch.Size([5376, 65160, 7])


In [9]:
# Load model and run one forward pass
from model.hi_gnn import HiGNN

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using: {device}')

# Load graph
def load_global_graph(graph_dir, device):
    m2m_edge_index = torch.load(f'{graph_dir}/m2m_edge_index.pt', map_location=device)
    num_levels = len(m2m_edge_index)
    graph = {
        'g2m_edge_index':  torch.load(f'{graph_dir}/g2m_edge_index.pt', map_location=device),
        'g2m_features':    torch.load(f'{graph_dir}/g2m_features.pt',   map_location=device),
        'm2g_edge_index':  torch.load(f'{graph_dir}/m2g_edge_index.pt', map_location=device),
        'm2g_features':    torch.load(f'{graph_dir}/m2g_features.pt',   map_location=device),
        'm2m_edge_index':  m2m_edge_index,
        'm2m_features':    torch.load(f'{graph_dir}/m2m_features.pt',   map_location=device),
        'up_edge_index':   torch.load(f'{graph_dir}/mesh_up_edge_index.pt',   map_location=device),
        'up_features':     torch.load(f'{graph_dir}/mesh_up_features.pt',     map_location=device),
        'down_edge_index': torch.load(f'{graph_dir}/mesh_down_edge_index.pt', map_location=device),
        'down_features':   torch.load(f'{graph_dir}/mesh_down_features.pt',   map_location=device),
        'mesh_features':   torch.load(f'{graph_dir}/mesh_features.pt',        map_location=device),
    }
    return graph, num_levels

graph, num_levels = load_global_graph(DATA_DIR, device)
edge_dim = graph['g2m_features'].shape[1]

model = HiGNN(node_dim=7, edge_dim=edge_dim, num_levels=num_levels).to(device)

# Try loading checkpoint if it exists
ckpt_path = f'{DATA_DIR}/tmodel.pt'
print(f'Looking for checkpoint at: {ckpt_path}')
print(f'Exists: {os.path.exists(ckpt_path)}')
if os.path.exists(ckpt_path):
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    print('Loaded checkpoint')
else:
    print('No checkpoint found — using random weights (visualization will be noise)')

model.eval()

Using: cuda
Looking for checkpoint at: data/global/tmodel.pt
Exists: True
Loaded checkpoint


HiGNN(
  (grid_encoder): Linear(in_features=7, out_features=128, bias=True)
  (g2m_gnn): MessagePassingLayer(
    (message_mlp): Sequential(
      (0): Linear(in_features=131, out_features=128, bias=True)
      (1): ReLU()
      (2): Linear(in_features=128, out_features=128, bias=True)
    )
    (update_mlp): Sequential(
      (0): Linear(in_features=256, out_features=128, bias=True)
      (1): ReLU()
      (2): Linear(in_features=128, out_features=128, bias=True)
    )
  )
  (m2g_gnn): MessagePassingLayer(
    (message_mlp): Sequential(
      (0): Linear(in_features=131, out_features=128, bias=True)
      (1): ReLU()
      (2): Linear(in_features=128, out_features=128, bias=True)
    )
    (update_mlp): Sequential(
      (0): Linear(in_features=256, out_features=128, bias=True)
      (1): ReLU()
      (2): Linear(in_features=128, out_features=128, bias=True)
    )
  )
  (grid_decoder): Linear(in_features=128, out_features=7, bias=True)
  (same_gnns): ModuleList(
    (0-2): 3 x Message

In [10]:
# Run one forward pass
node_features = torch.load(f'{DATA_DIR}/node_features.pt').to(device)
test_data = node_features[4032:]  # test split

t = 0
x = test_data[t]  # (N_grid, 7)
target = test_data[t + 1]

with torch.no_grad():
    delta = model(x, graph)
    pred = x + delta

# Compute per-node absolute error for t850
var_idx = 3  # t850
error = torch.abs(pred[:, var_idx] - target[:, var_idx]).cpu().numpy()

print(f'Error shape: {error.shape}')
print(f'Mean error: {error.mean():.4f}')
print(f'Max error:  {error.max():.4f}')

Error shape: (65160,)
Mean error: 0.0125
Max error:  0.2058


In [11]:
# 3D elevated error globe
fig = plot_globe_elevated(
    error=error,
    lat=lat, lon=lon,
    title=f't850 Absolute Error — Elevated Globe (T+1)',
    elevation_scale=0.4,
    colorscale='Reds'
)
fig.show()

In [12]:
# All variables side by side — flat colored globe
pred_cpu = pred.cpu().numpy()
target_cpu = target.cpu().numpy()

for var_idx, var_name in enumerate(VAR_NAMES):
    err = np.abs(pred_cpu[:, var_idx] - target_cpu[:, var_idx])
    
    fig = plot_globe_elevated(
        error=err,
        lat=lat, lon=lon,
        title=f'{var_name} — Error Globe (T+1)',
        elevation_scale=0.3,
        colorscale='plasma'
    )
    fig.show()
    break  # remove break to show all variables

In [13]:
var_idx = 3  # t850
var_name = VAR_NAMES[var_idx]

# Denormalize
pred_denorm   = pred_cpu[:, var_idx] * std[var_idx] + mean[var_idx]
target_denorm = target_cpu[:, var_idx] * std[var_idx] + mean[var_idx]

fig_actual = plot_globe_scatter(
    values=target_denorm,
    lat=lat, lon=lon,
    title=f'{var_name} — Actual (K)',
    colorscale='RdBu_r',
    symmetric=False
)
fig_actual.show()

fig_pred = plot_globe_scatter(
    values=pred_denorm,
    lat=lat, lon=lon,
    title=f'{var_name} — Predicted (K)',
    colorscale='RdBu_r',
    symmetric=False
)
fig_pred.show()